## DNA Transcription/Translation

Proteins are formed in the body first by transcription of DNA to mRNA, and then translation of mRNA to polymers, which are chains of amino acids. Let's see this first with some code:

First, we define a function for transcription: taking a DNA template strand and transcribing it into its corresponding mRNA strand. For example, if we gave it `TAC`, it would return `AUG`.

In [ ]:
def transcribe(dna: str) -> str:
    mapping = {
        'A': 'U',
        'T': 'A',
        'G': 'C',
        'C': 'G',
    }
    return ''.join(mapping[b] for b in dna)

We'll define another function for translation: taking an mRNA strand and translating it into its correspdoning polymer. For example, if we gave it `AUGUUA`, we would get `['Met', Leu']`.

In [ ]:
# mRNA codon table
# Maps codons to their corresponding amino acids
# Each line belonds to one amino acid
codon_to_aa = {
    'AUG': 'Met',  # start
    'UUU': 'Phe', 'UUC': 'Phe',
    'UUA': 'Leu', 'UUG': 'Leu',
    'UAA': 'Stop', 'UAG': 'Stop', 'UGA': 'Stop'
}


# mRNA -> protein translation
def translate(mrna: str):
    protein = []

    # Iterate through the mRNA strand in chunks of three
    for i in range(0, len(mrna), 3):
        codon = mrna[i:i+3]
        if len(codon) < 3:
            break

        aa = codon_to_aa.get(codon, '?')  # '?' for unlisted codons

        if aa == 'Stop':
            break

        protein.append(aa)

    return protein


Here's an example usage.

In [ ]:
dna = "TACAAAAATATT"

mrna = transcribe(dna)
protein = translate(mrna)

print("DNA:     ", dna)
print("mRNA:    ", mrna)
print("Protein: ", protein)

## Reverse DNA Transcription/Translation

Notice how each amino acid can be translated from multiple different codons. If we were given a certain protein, how could be reverse the process of transcription and translation to recover the original DNA template strand that created the protein? Let's go through this step by step.

First, we'll declare what we're given.

In [ ]:
# Given protein sequence
protein = ["Met", "Phe", "Stop"]

# Some example amino acids and what codons they translate from
aa_to_codons = {
    "Met":  ["AUG"],
    "Phe":  ["UUU", "UUC"],
    "Stop": ["UAA", "UAG", "UGA"],
}

Next, we'll set up our solver and its constraints.

In [ ]:
def solve_protein_to_dna(protein, reverse_codon_table, n=5):

    # Encoding each base as an integer for Z3
    A, U, G, C = 0, 1, 2, 3
    base_to_num = {"A": A, "U": U, "G": G, "C": C}
    num_to_base = {A: "A", U: "U", G: "G", C: "C"}

    s = Solver()

    # For each amino acid,
    # we want to define three Z3 variables: one for each mRNA base.
    # `mrna` holds these codon variables
    # mrna = [Int(f"mrna_{i}") for i in range(3 * len(protein))]
    mrna = [
        [Int(f"mrna_{i}_1"), Int(f"mrna_{i}_2"), Int(f"mrna_{i}_3")]
        for i in range(len(protein))
    ]

    # Each mRNA base must be A, U, G, or C
    for codon in mrna:
        for b in codon:
            s.add(Or(b == A, b == U, b == G, b == C))

    # Add codon constraints
    for i, aa in enumerate(protein):
        # Grab the ith codon, destructure into its three bases
        b1, b2, b3 = mrna[i]

        choices = []

        # Important part!!!
        for codon in reverse_codon_table[aa]:
            choices.append(
                And(
                    b1 == base_to_num[codon[0]],
                    b2 == base_to_num[codon[1]],
                    b3 == base_to_num[codon[2]],
                )
            )

        s.add(Or(choices))

    # Solve
    if s.check() == sat:
        print("Protein:     ", protein)
        print()

        # Print first n solutions that the solver found. n = 5 by default.
        n = 5
        i = 0
        while s.check() == sat and i < n:
            model = s.model()

            mrna_seq = ""

            for codon in mrna:
                mrna_seq += "".join(num_to_base[model[b].as_long()] for b in codon)

            # Reverse transcribe mRNA -> DNA template
            mrna_to_dna = {"A": "T", "U": "A", "G": "C", "C": "G"}

            dna_template = "".join(mrna_to_dna[b] for b in mrna_seq)

            print("mRNA:        ", mrna_seq)
            print("DNA template:", dna_template)
            print()

            # Block current solution
            s.add(Or([b != model[b] for codon in mrna for b in codon]))

            # Increment counter
            i += 1
    else:
        print("No solution")

solve_protein_to_dna(protein, aa_to_codons)

For fun, here's the same solver code above being used on a much larger codon table.

In [ ]:
# Given protein sequence
protein = ["Met", "Gly", "Ser", "Lys", "Stop"]

CODON_TABLE = {
    "UUU": "Phe", "UUC": "Phe", "UUA": "Leu", "UUG": "Leu",
    "UCU": "Ser", "UCC": "Ser", "UCA": "Ser", "UCG": "Ser",
    "UAU": "Tyr", "UAC": "Tyr", "UAA": "Stop", "UAG": "Stop",
    "UGU": "Cys", "UGC": "Cys", "UGA": "Stop", "UGG": "Trp",
    "CUU": "Leu", "CUC": "Leu", "CUA": "Leu", "CUG": "Leu",
    "CCU": "Pro", "CCC": "Pro", "CCA": "Pro", "CCG": "Pro",
    "CAU": "His", "CAC": "His", "CAA": "Gln", "CAG": "Gln",
    "CGU": "Arg", "CGC": "Arg", "CGA": "Arg", "CGG": "Arg",
    "AUU": "Ile", "AUC": "Ile", "AUA": "Ile", "AUG": "Met",
    "ACU": "Thr", "ACC": "Thr", "ACA": "Thr", "ACG": "Thr",
    "AAU": "Asn", "AAC": "Asn", "AAA": "Lys", "AAG": "Lys",
    "AGU": "Ser", "AGC": "Ser", "AGA": "Arg", "AGG": "Arg",
    "GUU": "Val", "GUC": "Val", "GUA": "Val", "GUG": "Val",
    "GCU": "Ala", "GCC": "Ala", "GCA": "Ala", "GCG": "Ala",
    "GAU": "Asp", "GAC": "Asp", "GAA": "Glu", "GAG": "Glu",
    "GGU": "Gly", "GGC": "Gly", "GGA": "Gly", "GGG": "Gly",
}

# Reverse mapping
from collections import defaultdict
REVERSE_CODON_TABLE = defaultdict(list)

for codon, aa in CODON_TABLE.items():
    REVERSE_CODON_TABLE[aa].append(codon)

# Solve it!
solve_protein_to_dna(protein, REVERSE_CODON_TABLE)